# Waterline — U-Net baseline (issue #8)

Trains `FloodUNet` (VV+VH, resnet34) on MMFlood and scores it on the real event-wise test split, for comparison against the Otsu classical baseline (#7).

**Run this on a GPU runtime**: Runtime > Change runtime type > T4 GPU (or better).

Repo: https://github.com/sanjayBahadur/waterline — see `CLAUDE.md` for the project's operating rules and `.userlogs.txt`-adjacent PR history for why each choice below was made.

**Nothing in this notebook stores the dataset in the repo.** Data is fetched fresh into this Colab session's own disk, every run — the same manual, explicit-download convention the rest of the project follows (`CLAUDE.md` "Data handling": acquisition is never automated inside library code, only ever a visible, user-triggered step — this notebook cell is that step, for this environment).

## 1. Install

Installs the `waterline` package straight from GitHub (pulls in its pinned dependencies — torch, torchgeo, lightning, segmentation-models-pytorch, kornia — via `pyproject.toml`), plus the two things this notebook adds that the core package doesn't need: `geemap` (the visualization step) and `wandb` (training logging; usually already present on Colab, pinned here for certainty).

In [ ]:
%%capture
!pip install "git+https://github.com/sanjayBahadur/waterline.git"
!pip install geemap wandb

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected -- go to Runtime > Change runtime type and select a GPU before continuing."
    )

## 2. Stage the data

MMFlood is ~47GB (11 tar parts). This downloads into Colab's own local disk -- not into your Google Drive, and not into the repo -- so it's gone when this session ends and gets pulled fresh next time. That's intentional: the dataset never touches GitHub.

In [ ]:
from pathlib import Path

from torchgeo.datasets import MMFlood

DATA_ROOT = Path("/content/data/mmflood")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# download=True here is deliberate and explicit -- see the note at the top
# of this notebook. Just confirming the train split downloads; val/test get
# pulled the first time WaterlineDataModule touches them below.
_ = MMFlood(root=str(DATA_ROOT), split="train", include_dem=False, download=True)
print(f"MMFlood staged at {DATA_ROOT}")

## 3. Authenticate

Both of these open an interactive prompt the first time you run them in a session -- follow the link, paste the key/code back in.

In [ ]:
import wandb

wandb.login()

In [ ]:
import ee

# Needs a free Earth Engine account (https://earthengine.google.com/) --
# sign up once if you haven't. Replace with your own GEE-registered
# Cloud project id.
EE_PROJECT = "your-gee-project-id"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

## 4. Data module and model

`include_dem=False`, `split_strategy="event"` -- this is the M1 baseline (VV+VH only, honest event-wise split). M2's terrain ablation is the only place DEM gets added, as a clean, isolated change against this exact setup (`CLAUDE.md` hard rule #5).

In [ ]:
from waterline.data.mmflood import WaterlineDataModule
from waterline.models.unet import FloodUNet

PATCH_SIZE = 256
BATCH_SIZE = 16  # tune to your GPU's memory if this OOMs

dm = WaterlineDataModule(
    data_root=DATA_ROOT,
    split_strategy="event",
    include_dem=False,
    patch_size=PATCH_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=2,
)
dm.setup()
print(
    f"train={len(dm.train_dataset)} val={len(dm.val_dataset)} test={len(dm.test_dataset)} tiles"
)

model = FloodUNet(encoder_name="resnet34", encoder_weights="imagenet", lr=1e-4)

## 5. Train

`max_epochs=30` is a starting point, not a tuned final number -- watch the W&B curves and adjust. `EarlyStopping`/`ModelCheckpoint` are standard Lightning conveniences, not modeling decisions. The checkpoint saves to Colab's local disk -- **download it or copy it to Drive before the session ends**, or it's lost.

In [ ]:
import lightning as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger

from waterline.models.unet import QualitativePanelLogger

wandb_logger = WandbLogger(project="waterline", name="unet-baseline-vvvh")
checkpoint_cb = ModelCheckpoint(
    monitor="val_iou",
    mode="max",
    save_top_k=1,
    filename="unet-baseline-{epoch}-{val_iou:.3f}",
)

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1,
    logger=wandb_logger,
    callbacks=[
        checkpoint_cb,
        EarlyStopping(monitor="val_iou", mode="max", patience=5),
        QualitativePanelLogger(n_samples=4),
    ],
)
trainer.fit(model, datamodule=dm)

print(f"Best checkpoint: {checkpoint_cb.best_model_path}")

## 6. Evaluate on the real test split

Uses the same harness that scored Otsu (#7), so the numbers are directly comparable -- same metric definitions, same per-event and per-terrain-relief breakdown, same test events.

**Otsu's numbers, for reference (test split, 34 events, measured 2026-09-28):**

| | IoU | Precision | Recall | F1 |
|---|---|---|---|---|
| Overall | 0.036 | 0.036 | 0.968 | 0.070 |
| Flat terrain | 0.065 | 0.065 | -- | -- |
| High-relief terrain | 0.019 | 0.019 | -- | -- |

In [ ]:
import torch

from waterline.eval.harness import evaluate_subset

model.eval()
device = next(model.parameters()).device


def unet_predict(image: torch.Tensor) -> torch.Tensor:
    """Wraps the trained model into the harness's predict_fn signature:
    one (C, H, W) patch in, one (H, W) 0/1 prediction out.
    """
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits).squeeze(0).squeeze(0) > 0.5).long()
    return pred.cpu()


report = evaluate_subset(
    unet_predict, DATA_ROOT, subset="test", include_dem=False, patch_size=PATCH_SIZE
)
print("OVERALL:", report.overall)
print("PER RELIEF:")
for bucket, metrics in report.per_relief.items():
    print(f"  {bucket}: {metrics}")

## 7. Visualize a prediction on a real map

Renders one test patch as an actual interactive, georeferenced map -- the real Sentinel-1 imagery as the basemap (fetched live from Earth Engine's own Sentinel-1 archive for the same tile's location/date), with the model's prediction and the ground truth as overlay layers you can toggle.

In [ ]:
import geemap
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from torchgeo.samplers import GridGeoSampler

from waterline.data.mmflood import event_ids_for_subset, split_for_events

# Grab one real test-split patch and its prediction to visualize.
test_events = event_ids_for_subset(DATA_ROOT, "test")
sample_ds = split_for_events(DATA_ROOT, test_events[:1], include_dem=False)
sampler = GridGeoSampler(sample_ds, size=PATCH_SIZE, stride=PATCH_SIZE)
query = next(iter(sampler))  # (x-slice, y-slice, t-slice) -- verified against real data
sample = sample_ds[
    query
]  # NOT sample_ds.bounds, which is the whole *event's* extent, not this one patch
pred = unet_predict(sample["image"])

xmin, xmax = query[0].start, query[0].stop
ymin, ymax = (
    query[1].start,
    query[1].stop,
)  # sample_ds.crs is EPSG:4326 (lat/lon) -- verified, not a projected CRS
height, width = sample["mask"].shape
transform = from_bounds(xmin, ymin, xmax, ymax, width, height)


def _write_local_geotiff(path: str, array: np.ndarray, dtype: str) -> None:
    """Writes one band as a small georeferenced GeoTIFF, positioned at this
    patch's real coordinates -- geemap.Map.add_raster reads these directly,
    no Earth Engine asset upload needed for our own prediction/ground-truth
    layers (only the Sentinel-1 basemap below comes from Earth Engine).
    """
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=dtype,
        crs=str(sample_ds.crs),
        transform=transform,
    ) as dst:
        dst.write(array, 1)


_write_local_geotiff("/content/pred.tif", pred.numpy().astype("uint8"), "uint8")
ground_truth = sample["mask"].numpy()
_write_local_geotiff(
    "/content/ground_truth.tif",
    np.where(ground_truth == 1, 1, 0).astype("uint8"),
    "uint8",
)

region = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

m = geemap.Map()
m.centerObject(region, zoom=13)

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(region)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .first()
)
m.addLayer(
    s1.select("VV"), {"min": -25, "max": 0}, "Sentinel-1 VV (live from Earth Engine)"
)
m.add_raster(
    "/content/ground_truth.tif",
    palette=["00000000", "0089A3"],
    layer_name="Ground truth",
)
m.add_raster(
    "/content/pred.tif", palette=["00000000", "C1600F"], layer_name="Model prediction"
)
m